# Feature Engineering pour Arrivals et Nights

In [1]:
import pandas as pd
import numpy as np
import os

file_path = '../data/merged_tourism_data_final.csv'
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'])

targets = ['Arrivals', 'Nights']
folder_path = '../data/separted'
os.makedirs(folder_path, exist_ok=True)

for target in targets:
    print(f"--- Feature Engineering pour {target} ---")
    df_t = df.copy()
    
    # Lags
    lags = [1 , 2 ,6 ,12]
    for k in lags:
        df_t[f'lags_{k}'] = df_t[target].shift(k)
        
    # Rolling
    windows = [3 , 6, 12]
    for w in windows :
        df_t[f'roll_mean_{w}'] = df_t[target].shift(1).rolling(window=w).mean()
        df_t[f'roll_std_{w}'] = df_t[target].shift(1).rolling(window=w).std()

    # Growth
    df_t['growth_yoy'] = (df_t[target] - df_t[target].shift(12)) / df_t[target].shift(12) * 100
    df_t.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Time
    df_t['month'] = df_t['Date'].dt.month
    df_t['year'] = df_t['Date'].dt.year
    df_t['quarter'] = df_t['Date'].dt.quarter
    df_t['month_sin'] = np.sin(2 * np.pi * df_t['month'] / 12)
    df_t['month_cos'] = np.cos(2 * np.pi * df_t['month'] / 12)
    
    def get_season(m):
        if m in [12, 1, 2]: return 0
        elif m in [3, 4, 5]: return 1
        elif m in [6, 7, 8]: return 2
        else: return 3
    df_t['season'] = df_t['month'].apply(get_season)
    df_t = pd.get_dummies(df_t, columns=['season'], prefix='saison', drop_first=True)
    
    # Eco
    eco_vars = ['Oil_price', 'FDI', 'Poverty_rate', 'GDP_Construction']
    for var in eco_vars:
        if var in df_t.columns:
            df_t[f'{var}_lag1'] = df_t[var].shift(1)
            df_t[f'{var}_lag3'] = df_t[var].shift(3)
    if 'REER' in df_t.columns:
        df_t['REER_lag1'] = df_t['REER'].shift(1)
        
    # Events
    df_t['cdm_event'] = 0
    df_t.loc[(df_t['Date'] >= '2025-12-01') & (df_t['Date'] <= '2026-01-31'), 'cdm_event'] = 1
    df_t.loc[(df_t['Date'] >= '2030-06-01') & (df_t['Date'] <= '2030-07-31'), 'cdm_event'] = 1
    
    FEATURES_ML = [
        'lags_1', 'lags_2', 'lags_12',
        'roll_mean_3', 'roll_mean_6', 'roll_mean_12',
        'roll_std_3', 'roll_std_6', 'roll_std_12', 'growth_yoy',
        'month_sin', 'month_cos', 'quarter', 'year',
        'saison_1', 'saison_2', 'saison_3',
        'Oil_price_lag1', 'FDI_lag1', 'Poverty_rate_lag1', 'REER_lag1',
        'is_covid', 'cdm_event'
    ]
    
    df_ml = df_t.dropna(subset=[target]).copy()
    valid_features = [c for c in FEATURES_ML if c in df_ml.columns]
    
    train_end = '2022-12-31'
    test_start = '2023-01-01'
    
    X_train = df_ml[df_ml['Date'] <= train_end][valid_features]
    y_train = df_ml[df_ml['Date'] <= train_end][target]
    
    X_test = df_ml[df_ml['Date'] >= test_start][valid_features]
    y_test = df_ml[df_ml['Date'] >= test_start][target]
    
    X_train.to_csv(f'{folder_path}/X_train_{target}.csv', index=False)
    y_train.to_csv(f'{folder_path}/y_train_{target}.csv', index=False)
    X_test.to_csv(f'{folder_path}/X_test_{target}.csv', index=False)
    y_test.to_csv(f'{folder_path}/y_test_{target}.csv', index=False)
    
    print(f"Features extraites pour {target} : Train size = {len(X_train)}, Test size = {len(X_test)}")


--- Feature Engineering pour Arrivals ---
Features extraites pour Arrivals : Train size = 336, Test size = 40
--- Feature Engineering pour Nights ---
Features extraites pour Nights : Train size = 11, Test size = 38
